# 🚀 DBT Snapshots --- Complete Guide (With YAML + SQL Examples)

------------------------------------------------------------------------

# 🧠 1. What are DBT Snapshots?

👉 DBT Snapshots track **historical changes in data (SCD Type 2)**

-   Stores previous versions of records\
-   Adds time-based validity\
-   Helps in auditing & historical analysis

------------------------------------------------------------------------

# 🔥 Why Snapshots?

👉 Problem: Source data gets overwritten

``` text
customer_id | status
------------|--------
1           | active
```

Later:

``` text
1 | inactive
```

❌ Old data lost

------------------------------------------------------------------------

## ✅ Snapshot Output

``` text
customer_id | status   | dbt_valid_from | dbt_valid_to
------------|----------|----------------|--------------
1           | active   | t1             | t2
1           | inactive | t2             | NULL
```

------------------------------------------------------------------------

# ⚙️ Snapshot Strategies

------------------------------------------------------------------------

## 1. Timestamp Strategy (Recommended)

-   Uses `updated_at` column\
-   Efficient & reliable

------------------------------------------------------------------------

## 2. Check Strategy

-   Compares column values\
-   Used when no timestamp available

------------------------------------------------------------------------

# 🧩 2. Snapshot SQL Example (Timestamp)

------------------------------------------------------------------------

``` sql
-- snapshots/customer_snapshot.sql

{% snapshot customer_snapshot %}

{{
    config(
      target_schema='snapshots',
      unique_key='customer_id',
      strategy='timestamp',
      updated_at='updated_at'
    )
}}

SELECT
    customer_id,
    name,
    status,
    updated_at
FROM {{ source('raw', 'customers') }}

{% endsnapshot %}
```

------------------------------------------------------------------------

# 🧩 3. Snapshot YAML Configuration

------------------------------------------------------------------------

## 📁 snapshots/snapshots.yml

``` yaml
snapshots:
  - name: customer_snapshot
    config:
      target_schema: snapshots
      unique_key: customer_id
      strategy: timestamp
      updated_at: updated_at
```

------------------------------------------------------------------------

# 🧩 4. Check Strategy Example

------------------------------------------------------------------------

``` sql
{% snapshot product_snapshot %}

{{
    config(
      target_schema='snapshots',
      unique_key='product_id',
      strategy='check',
      check_cols=['price', 'category']
    )
}}

SELECT *
FROM {{ source('raw', 'products') }}

{% endsnapshot %}
```

------------------------------------------------------------------------

## YAML Version

``` yaml
snapshots:
  - name: product_snapshot
    config:
      unique_key: product_id
      strategy: check
      check_cols: ['price', 'category']
```

------------------------------------------------------------------------

# ▶️ Run Snapshots

``` bash
dbt snapshot
```

------------------------------------------------------------------------

# 🧠 Internal Working

1.  Reads current data\
2.  Compares with existing snapshot\
3.  Detects changes\
4.  Inserts new version\
5.  Updates dbt_valid_to

------------------------------------------------------------------------

# 🧾 Snapshot Output Columns

  Column           Description
  ---------------- ----------------
  dbt_valid_from   start time
  dbt_valid_to     end time
  dbt_updated_at   last update
  dbt_scd_id       unique version

------------------------------------------------------------------------

# 🏗️ Real Production Example

------------------------------------------------------------------------

## 🎯 Use Case: Track Subscription Changes

------------------------------------------------------------------------

### Step 1: Snapshot SQL

``` sql
{% snapshot subscription_snapshot %}

{{
    config(
      unique_key='user_id',
      strategy='timestamp',
      updated_at='updated_at'
    )
}}

SELECT
    user_id,
    plan,
    status,
    updated_at
FROM {{ source('raw', 'subscriptions') }}

{% endsnapshot %}
```

------------------------------------------------------------------------

### Step 2: YAML Config

``` yaml
snapshots:
  - name: subscription_snapshot
    config:
      unique_key: user_id
      strategy: timestamp
      updated_at: updated_at
```

------------------------------------------------------------------------

### Step 3: Use Snapshot

``` sql
SELECT *
FROM {{ ref('subscription_snapshot') }}
WHERE dbt_valid_to IS NULL
```

------------------------------------------------------------------------

# ⚡ When to Use Snapshots

------------------------------------------------------------------------

## ✅ Use When:

-   Need historical tracking\
-   Slowly changing dimensions\
-   Audit logs

------------------------------------------------------------------------

## ❌ Avoid When:

-   Data rarely changes\
-   Real-time streaming needed

------------------------------------------------------------------------

# 🚀 Best Practices

-   Prefer timestamp strategy\
-   Ensure updated_at is reliable\
-   Keep snapshot tables optimized\
-   Partition large tables

------------------------------------------------------------------------

# 🧠 Snapshot vs Incremental

  Feature         Snapshot          Incremental
  --------------- ----------------- ---------------
  History         Yes               No
  Use Case        SCD               Performance
  Data Handling   Append versions   Update/insert

------------------------------------------------------------------------

# 🎯 Interview Points

-   Snapshots implement SCD Type 2\
-   YAML + SQL both can define config\
-   Timestamp strategy preferred\
-   Stores full history

------------------------------------------------------------------------

# ⚡ Final Summary

DBT Snapshots: - Track changes over time\
- Preserve history\
- Enable audit & analytics

👉 Core Idea: Snapshots = Historical versioning of data
